# Stage 1 + Stage 2 Pipeline (per DOE)

Process every test item in one DOE folder in order:
1. **Stage 1**: Convert each test source workbook into `stage1_<test_item>/summary_features.parquet`.
2. **Stage 2**: Automatically merge all Stage 1 outputs into `stage2_<DOE>.xlsx` for review plus `.parquet` for downstream ingestion.

### Test Type Selection
- `cycling`: RT 1C1C, HT 1C1C, RT 4C1C, HT 1_5C3C, etc. Use a single-test feature analysis workbook.
- `ht_storage`: Use an HT storage summary workbook.
- `rate_cap`: Use a `*Rate_discharge*.xlsx` workbook that includes per-cell sheets.
- `formation`: Use a `*color_coded_Summary.xlsx` workbook plus electrode area.


In [ ]:
# Cell 1.  Imports + locate modules
import os, sys
from pathlib import Path
import pandas as pd
import traitlets
from ipywidgets import widgets
from IPython.display import display, clear_output
from tkinter import Tk, filedialog

# Locate the helper modules
SEARCH_DIRS = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd(),
]
for d in SEARCH_DIRS:
    if (d / "stage1_extractor.py").exists() and (d / "stage2_aggregator.py").exists():
        sys.path.insert(0, str(d))
        MOD_DIR = d
        break
else:
    raise FileNotFoundError("stage1_extractor.py / stage2_aggregator.py not found")

from stage1_extractor import run_stage1
from stage2_aggregator import run_stage2, discover_stage1_folders
print(f"modules loaded from {MOD_DIR}")


In [ ]:
# Cell 2.  Pick DOE root folder
class DirPickButton(widgets.Button):
    def __init__(self, label):
        super().__init__()
        self.add_traits(path=traitlets.Unicode())
        self.description = label
        self.style.button_color = "orange"
        self.on_click(self._on_click)
    def _on_click(self, b):
        with out_doe:
            clear_output()
            try:
                root = Tk(); root.withdraw()
                root.call("wm","attributes",".","-topmost",True)
                p = filedialog.askdirectory(title="Select DOE root folder")
                if p:
                    b.path = p
                    doe_text.value = p
                    found = discover_stage1_folders(p)
                    print(f"DOE root: {p}")
                    if found:
                        print("Existing stage1 folders:")
                        for k,v in found.items(): print(f"  {k}")
                    else:
                        print("(no stage1_* subfolders yet — run Stage 1 below)")
            except Exception as e:
                print("error:", e)

out_doe = widgets.Output()
doe_btn = DirPickButton("Select DOE root folder")
doe_text = widgets.Text(description="DOE root:", layout=widgets.Layout(width="900px"))
doe_name_text = widgets.Text(description="DOE name:", value="",
                              placeholder="(blank = use folder name)",
                              layout=widgets.Layout(width="500px"))
display(widgets.VBox([doe_btn, doe_text, doe_name_text, out_doe]))


In [ ]:
# Cell 3.  Stage 1 — pick all test items at once, then click "Run all"
# Each row is OPTIONAL: leave xlsx empty to skip that test.
import os
from tkinter import Tk, filedialog
from ipywidgets import widgets
from IPython.display import display, clear_output
import traitlets

class _PickFile(widgets.Button):
    def __init__(self, target):
        super().__init__()
        self.description = "📁"; self.layout = widgets.Layout(width="50px")
        self.style.button_color = "orange"
        self._t = target
        self.on_click(self._click)
    def _click(self, b):
        try:
            r = Tk(); r.withdraw(); r.call("wm","attributes",".","-topmost",True)
            p = filedialog.askopenfilename(title="Source xlsx",
                                            filetypes=[("Excel","*.xlsx *.xlsm")])
            r.destroy()
            if p:
                self._t.value = p
                self.style.button_color = "lightgreen"
                self.description = "✓"
        except Exception as e: print("err:", e)

# 6 rows: 3 cycling + ht_storage + rate_cap + formation
def _row(test_type, default_name, default_label, extra=None):
    name = widgets.Text(value=default_name, description="name:",
                        layout=widgets.Layout(width="200px"),
                        style={"description_width":"50px"})
    path = widgets.Text(description="xlsx:", layout=widgets.Layout(width="700px"),
                        style={"description_width":"50px"},
                        placeholder="(leave empty to SKIP)")
    btn = _PickFile(path)
    items = [widgets.HTML(f"<b style='color:#666'>{default_label}</b>"),
             widgets.HBox([name, btn, path])]
    if extra is not None: items.append(extra)
    return {"type": test_type, "name": name, "path": path, "extra": extra,
            "box": widgets.VBox(items)}

rows = []
rows.append(_row("cycling",   "RT_1C1C",      "Cycling — RT 1C1C"))
rows.append(_row("cycling",   "HT_1C1C",      "Cycling — HT 1C1C"))
rows.append(_row("cycling",   "HT_1_5C_3C",   "Cycling — HT 1.5C/3C"))
rows.append(_row("ht_storage","HT_storage",   "HT Storage"))
rows.append(_row("rate_cap",  "rate_cap",     "Rate Capability"))
# Formation needs electrode area
fa = widgets.FloatText(value=12.5, description="area (cm²):",
                       layout=widgets.Layout(width="280px"),
                       style={"description_width":"110px"})
fa_regex = widgets.Text(value=r"^CELL0(0\d|1[0-8])\d{3}$",
                        description="DOE barcode regex:",
                        layout=widgets.Layout(width="500px"),
                        style={"description_width":"160px"})
rows.append(_row("formation","formation",     "Formation",
                 extra=widgets.HBox([fa, fa_regex])))

run_all_btn = widgets.Button(description="▶ Run all Stage 1",
                             style={"button_color":"lightgreen"},
                             layout=widgets.Layout(width="220px", height="40px"))
out_s1 = widgets.Output()

def _run_all(_):
    with out_s1:
        clear_output()
        doe = doe_text.value.strip()
        if not doe or not os.path.isdir(doe):
            print("Pick DOE root in Cell 2 first."); return
        from stage1_extractor import run_stage1
        n_done, n_skip = 0, 0
        for r in rows:
            xlsx = r["path"].value.strip()
            if not xlsx:
                print(f"  skip {r['name'].value} (no xlsx)"); n_skip += 1; continue
            if not os.path.isfile(xlsx):
                print(f"  ⚠ {r['name'].value}: file not found {xlsx}"); n_skip += 1; continue
            sub = os.path.join(doe, f"stage1_{r['name'].value}")
            os.makedirs(sub, exist_ok=True)
            print(f"\n>>> {r['name'].value}  ({r['type']})")
            try:
                kwargs = {}
                if r["type"] == "formation":
                    kwargs["electrode_area_cm2"]    = fa.value
                    kwargs["barcode_filter_regex"]  = fa_regex.value
                run_stage1(test_type=r["type"], xlsx_path=xlsx,
                           output_dir=sub, **kwargs)
                n_done += 1
            except Exception as e:
                import traceback
                print(f"   ✗ {r['name'].value} failed: {e}")
                traceback.print_exc()
                n_skip += 1
        print(f"\n=== Stage 1 done — {n_done} processed, {n_skip} skipped ===")

run_all_btn.on_click(_run_all)

display(widgets.VBox([r["box"] for r in rows] + [run_all_btn, out_s1]))

In [ ]:
# Cell 4.  Stage 2 — aggregate all stage1_* folders in the DOE root
baseline_text = widgets.Text(value="LOT.65",
                              description="baseline:",
                              style={"description_width":"160px"},
                              layout=widgets.Layout(width="500px"))
normalize_chk = widgets.Checkbox(value=True,
                                 description="normalize Electrolyte names (strip -A/-B/_v2 suffix)")

# Frozen scaler — paste the path that was created by the FIRST DOE of this chemistry/format
scaler_path_text = widgets.Text(
    value=str(Path.cwd() / "feature_scales_default.json"),
    description="scaler path:",
    style={"description_width":"160px"},
    layout=widgets.Layout(width="900px"))
first_doe_chk = widgets.Checkbox(
    value=False,
    description="This is the FIRST DOE for this chemistry/format → fit and save scaler now")

run_s2_btn = widgets.Button(description="Run Stage 2", style={"button_color":"lightgreen"})
out_s2 = widgets.Output()
result = {"obj": None}

def _run_stage2(b):
    with out_s2:
        clear_output()
        doe = doe_text.value.strip()
        if not doe or not os.path.isdir(doe):
            print("Pick DOE root first."); return
        try:
            r = run_stage2(
                doe_root=doe,
                baseline_electrolyte=baseline_text.value.strip(),
                output_dir=doe,
                doe_name=doe_name_text.value.strip() or None,
                scaler_path=scaler_path_text.value.strip() or None,
                refit_scaler_if_missing=bool(first_doe_chk.value),
                normalize_electrolyte_names=normalize_chk.value,
                verbose=True,
            )
            result["obj"] = r
            print()
            print("Stage 2 done.  Open the xlsx for human review:")
            print(f"  {r['output_xlsx']}")
            print(f"Parquet for downstream Stage 3:  {r['output_parquet']}")
        except Exception as e:
            import traceback; traceback.print_exc()
run_s2_btn.on_click(_run_stage2)

display(widgets.VBox([
    widgets.HTML("<b>Stage 2 — aggregate after all Stage 1 outputs are in place.</b>"),
    widgets.HTML("<i>Frozen scaler: the FIRST DOE of a chemistry/format fits + saves; all later DOEs load.</i>"),
    baseline_text, normalize_chk, scaler_path_text, first_doe_chk, run_s2_btn, out_s2,
]))


In [ ]:
# Cell 5.  Quick view — full pct_delta (INPUT + OUTPUT combined, color-coded)
if result["obj"] is None:
    print("Run Stage 2 first.")
else:
    import pandas as pd
    pd.set_option("display.float_format", "{:.2f}".format)
    print("=== Color-coded headline summary (image format) ===")
    display(result["obj"]["color_summary"])
    print()
    print("=== Full pct_delta vs baseline (all training features) ===")
    display(result["obj"]["pct_delta"])